# WTI Crude Oil Price Forecasting — Stateless Methods: Systematic Backtest (Notebook 4 of 7)

This notebook simulates a rigorous production forecasting workflow:

1. Run a **rolling weekly backtest across 2025** using
   `energy_oil_backtest.yaml` for all candidate predictors.
2. Compute metrics — **CRPS** for 5/10/21-day trajectories.
3. Select the **top contender configurations** based solely on 2025
   historical performance (no peeking at 2026).
4. Let the contenders compete in the **2026 Protected Arena**
   (`energy_oil_eval.yaml`) across the geopolitical price shock and its
   aftermath — measuring adaptive real-time responsiveness and calibration.
   The eval window runs through the most recent origin whose 21-business-day
   horizon still resolves against cached data (see `scripts/fetch_wti.py`).

The line-up spans three families behind one `Predictor` interface: **baselines**
(Naive, AutoARIMA), **numerical ML** (LightGBM ± a leak-safe covariate panel),
and **LLM/agent** methods (LLM-process forecasters and a news-reading analyst
agent) — the last run on *both* project models, `gemini-3.1-flash-lite-preview`
and `gemini-3.5-flash`. Every predictor is one toggle line in the registry in
Section 2. Agent configs come from `energy_oil_forecasting.analyst_agent`.

---
## 1. Setup, Data Registration & Spec Loading

In [1]:
import warnings
from pathlib import Path

from datetime import datetime
import energy_oil_forecasting
import pandas as pd
import yaml
from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    cached_multi_backtest,
    describe_spec,
)
from aieng.forecasting.models import ADVANCED_MODEL, LITE_MODEL
from energy_oil_forecasting.data import (
    DEFAULT_WTI_COVARIATE_SERIES_IDS,
    EXPANDED_WTI_COVARIATE_SERIES_IDS,
    LEVEL_VALUED_WTI_COVARIATE_SERIES_IDS,
    WTI_SERIES_ID,
    build_wti_multivariate_service,
)

from energy_oil_forecasting.analyst_agent.agent_original import (
    build_wti_news_config as build_wti_news_config_original,
    build_wti_agent_predictor as build_wti_agent_predictor_original,
)


warnings.filterwarnings("ignore")

# ── Mode ──────────────────────────────────────────────────────────────────────
# Set SMOKE_TEST = True to run a 2-origin, 1-sample version of the notebook
# for fast local development and end-to-end CI testing. The full specs run
# 51 backtest + 8 eval origins; smoke runs 2 + 2.
SMOKE_TEST = False 

# ── Models ────────────────────────────────────────────────────────────────────
# The project standardises on two Vector-proxy models. Every LLM and agent
# predictor below is run once per model so we can compare them head-to-head.
# (bare proxy names — no "gemini/" prefix)
MODELS = [LITE_MODEL, ADVANCED_MODEL]  # "gemini-3.1-flash-lite-preview", "gemini-3.5-flash"

# ── Derived settings (do not edit below) ─────────────────────────────────────
N_SAMPLES = 1 if SMOKE_TEST else 3  # trajectories per LLMP-Sampled call

# LightGBM hyperparameters (shared by the univariate and +covariate variants).
LAGS = 21  # one trading month of lagged target/covariate history
NUM_SAMPLES_LGBM = 100 if SMOKE_TEST else 200  # Monte-Carlo draws for quantiles
LGBM_KWARGS = {"num_threads": 1, "n_jobs": 1, "verbosity": -1}  # deterministic, quiet

# Data service: WTI target + a leak-safe covariate panel (all Yahoo Finance —
# Brent, natural gas, gasoline, gold, USD index, the USL/USO futures-curve
# contango proxy, and VIX). Non-covariate predictors simply ignore the extras,
# so one service feeds the whole leaderboard. Unavailable tickers are skipped
# with a warning, so this still runs offline / under partial connectivity.
# EIA_API_KEY, without putting it in this notebook. A VS Code kernel inherits
# VS Code's environment rather than the terminal's, so `export EIA_API_KEY=...`
# in a shell opened afterwards never reaches it -- which is why the two EIA
# series silently failed to build. Writing the key into a cell would fix that
# and then commit the key to GitHub the next time this notebook is pushed, so
# it is read from a gitignored file at the repo root instead:
#
#     echo 'your-key' > ~/agentic-forecasting/.eia_key
#
# Without either the env var or the file the EIA series are skipped and the
# panel is 14 rather than 16; the cell below says so explicitly.
import os
from pathlib import Path as _Path

if not os.environ.get("EIA_API_KEY"):
    for _candidate in (_Path.cwd(), *_Path.cwd().parents):
        _key_file = _candidate / ".eia_key"
        if _key_file.exists():
            os.environ["EIA_API_KEY"] = _key_file.read_text().strip()
            break

# Register the WIDER panel: the seven standard series plus OVX, the 3-2-1
# crack spread, and the 10-year yield. Registering a series does not put it
# in any model -- each predictor names its own list below -- so the existing
# predictors are untouched and their caches stay valid.
data_service = build_wti_multivariate_service(
    covariate_series_ids=EXPANDED_WTI_COVARIATE_SERIES_IDS
)
_available = set(data_service.series_ids)
COVARIATES = [c for c in DEFAULT_WTI_COVARIATE_SERIES_IDS if c in _available]
COVARIATES_EXPANDED = [c for c in EXPANDED_WTI_COVARIATE_SERIES_IDS if c in _available]
COVARIATES_LEVEL_ONLY = [c for c in LEVEL_VALUED_WTI_COVARIATE_SERIES_IDS if c in _available]

spec_dir = Path(energy_oil_forecasting.__file__).parent / "specs"
if SMOKE_TEST:
    backtest_file, eval_file = "energy_oil_smoke.yaml", "energy_oil_eval_smoke.yaml"
else:
    backtest_file, eval_file = "energy_oil_backtest.yaml", "energy_oil_eval.yaml"

with open(spec_dir / backtest_file) as f:
    backtest_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))
with open(spec_dir / eval_file) as f:
    eval_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))



# ── Extend to a 10yr backtest / 2yr protected holdout, quarterly cadence ────
# In SMOKE_TEST mode, use a small recent window instead so the notebook can
# be smoke-tested quickly. spec_id gets its own namespace so this never
# collides with (or silently reuses) the biweekly/original cached results.
_suffix = "_smoke" if SMOKE_TEST else ""

full_df = data_service.get_series(WTI_SERIES_ID, as_of=datetime.now()).sort_values("timestamp")
data_start = full_df["timestamp"].min()

# ── ANCHOR_END: the origin grid must not drift with the calendar ─────────────
# Everything below is derived from this one date, so taking it from
# full_df["timestamp"].max() made the whole grid a function of *when the cell
# ran*. Two runs a fortnight apart produced origins offset by ~10 business
# days — and because the prediction cache is keyed by spec_id alone (see
# aieng/forecasting/evaluation/artifacts.py: the window is neither part of the
# key nor checked on load), those mismatched grids landed in the same folder
# and were later compared as if they were the same experiment. They were not.
#
# 2026-07-23 is the value that reproduces the grid the cached agent runs in
# data/predictions/energy_oil_backtest_10yr_quarterly/ were actually computed
# on: backtest 2014-04-14 → 2024-07-23, eval 2024-07-23 → 2026-04-27.
# Change it only if you intend to invalidate that cache, and if you do, delete
# the cached files too — stale files are silently reloaded, not recomputed.
#
# (The separate 10yr cache from the local machine sits under the
# ..._localrun/ spec_id, which was run against ANCHOR_END = 2026-08-06. It is
# namespaced apart precisely so it can never be mistaken for this grid.)
ANCHOR_END = pd.Timestamp("2026-07-23")
data_end = full_df["timestamp"].max() if SMOKE_TEST else ANCHOR_END

holdout_start = data_end - pd.DateOffset(years=2)
holdout_end = data_end
backtest_start = data_start + (holdout_start - data_start) / 2
backtest_end = holdout_start

if SMOKE_TEST:
    backtest_spec.start = (data_end - pd.DateOffset(years=5)).strftime("%Y-%m-%d")
    backtest_spec.end = (data_end - pd.DateOffset(years=5) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    backtest_spec.stride = 5

    eval_spec.start = (data_end - pd.DateOffset(months=1)).strftime("%Y-%m-%d")
    eval_spec.end = data_end.strftime("%Y-%m-%d")
    eval_spec.stride = 5
else:
    backtest_spec.start = backtest_start.strftime("%Y-%m-%d")
    backtest_spec.end = backtest_end.strftime("%Y-%m-%d")
    backtest_spec.stride = 63

    eval_spec.start = holdout_start.strftime("%Y-%m-%d")
    eval_spec.end = (holdout_end - pd.tseries.offsets.BDay(63)).strftime("%Y-%m-%d")
    eval_spec.stride = 63

backtest_spec.tasks[0].horizons = [5, 10, 21, 63]
backtest_spec.spec_id = f"energy_oil_backtest_10yr_quarterly{_suffix}"

eval_spec.tasks[0].horizons = [5, 10, 21, 63]
eval_spec.spec_id = f"energy_oil_eval_10yr_quarterly{_suffix}"

# Guard: the cache cannot detect a grid mismatch, so assert the one we expect.
if not SMOKE_TEST:
    assert (backtest_spec.start, backtest_spec.end) == ("2014-04-14", "2024-07-23"), (
        f"Backtest grid drifted to {backtest_spec.start} → {backtest_spec.end}. "
        "The cached agent results were computed on 2014-04-14 → 2024-07-23; "
        "scoring against a different grid silently compares different experiments."
    )




print(f"{'⚡ SMOKE MODE' if SMOKE_TEST else '📊 FULL MODE'} — MODELS={MODELS}  N_SAMPLES={N_SAMPLES}")
# Report what actually registered, not what was asked for. build_wti_multivariate_service
# reports a failed fetch through warnings.warn, and this notebook's first line is
# warnings.filterwarnings("ignore") -- so a covariate that could not be built
# vanishes silently and the panel quietly shrinks. That matters most for the EIA
# pair, which is skipped outright when EIA_API_KEY is not visible to the kernel.
_missing = [c for c in EXPANDED_WTI_COVARIATE_SERIES_IDS if c not in _available]
print(f"Base panel  ({len(COVARIATES)}): {', '.join(COVARIATES) or '(none)'}")
print(f"Expanded    ({len(COVARIATES_EXPANDED)}/{len(EXPANDED_WTI_COVARIATE_SERIES_IDS)})")
print(f"Long-run-only ({len(COVARIATES_LEVEL_ONLY)})")
if _missing:
    print(f"!! {len(_missing)} requested covariate(s) FAILED to build and were dropped:")
    for _m in _missing:
        print(f"     {_m}")
    if any("_wl" in _m for _m in _missing):
        print("   The _wl series are EIA. Check EIA_API_KEY is set in the environment the")
        print("   KERNEL sees -- a VS Code kernel inherits VS Code's environment, not your")
        print("   terminal's, so an export after VS Code started will not reach it.")
else:
    print("all requested covariates built")
print()
print("━" * 72)
print("LOADED SPECIFICATIONS:")
print("━" * 72)
print(describe_spec(backtest_spec, data_service))
print(describe_spec(eval_spec, data_service))

📊 FULL MODE — MODELS=['gemini-3.1-flash-lite-preview', 'gemini-3.5-flash']  N_SAMPLES=3
Covariates registered (7): brent_log_ret_1b_l1b, natgas_log_ret_1b_l1b, gasoline_log_ret_1b_l1b, gold_log_ret_1b_l1b, dollar_index_log_ret_1b_l1b, oil_curve_contango_l1b, vix_level_l1b

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LOADED SPECIFICATIONS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MultiTargetBacktestSpec (spec_id=energy_oil_backtest_10yr_quarterly)
  description: Weekly rolling backtest in 2025 for daily WTI crude oil price forecasting. Evaluates trajectory forecasts (5, 10, 21 business days) with CRPS/MAE and binary up-shock forecasts (climb > $5 in 5 business days) with Brier Score. Used to select the top contender models.
  start:       2014-04-14
  end:         2024-07-23
  stride:      63
  warmup:      250
  tasks:       1

Task: wti_oil_price_forecast
  description: WTI Crude Oil continuous front-month futures Close pric

---
## 2. Candidate Predictors

This experiment puts a full slate of methods on the same `Predictor` interface and
the same rolling backtest, spanning three families:

| Family | Predictors | Role |
|---|---|---|
| **Baselines** | `Naive (Last Value)`, `AutoARIMA` | Carry-forward floor + the classical statistical anchor |
| **Numerical ML** | `LightGBM`, `LightGBM + cov` (+ optional `Prophet`) | Gradient-boosted quantile regression on lagged price (and a leak-safe covariate panel — Brent, gas, gasoline, gold, USD index, the futures-curve contango proxy, and VIX). LightGBM-with-covariates was the strongest method in the S&P 500 study. |
| **LLM / Agent** | `LLMP-Sampled`, `LLMP-Grid`, `News Agent` — each on **both** project models | LLM-process forecasters and a news-reading analyst agent, run on `gemini-3.1-flash-lite-preview` *and* `gemini-3.5-flash` |

The predictor cell below is a **registry**: every method is one line with an
`enabled` flag. Flip a flag to add or drop a predictor — the rest of the
notebook (backtest, scoring, eval, scorecard) iterates over whatever is active.
The two baselines are flagged `baseline=True` and are the only results written to
`adaptive_agent/curriculum/` for Notebooks 5–6, so toggling the others never
disturbs the downstream training data.

In [2]:
from dataclasses import dataclass
from typing import Callable

from aieng.forecasting.methods import (
    LastValuePredictor,
    QuantileGridLLMPredictor,
    QuantileGridLLMPredictorConfig,
    SampledTrajectoryLLMPredictor,
    SampledTrajectoryLLMPredictorConfig,
)
from aieng.forecasting.methods.numerical.darts_arima import DartsAutoARIMAPredictor
from aieng.forecasting.methods.numerical.darts_regression import DartsLightGBMPredictor
from aieng.forecasting.methods.numerical.error_correction_regression import (
    ErrorCorrectionRegressionPredictor,
)
from energy_oil_forecasting.analyst_agent import (
    build_wti_agent_predictor,
    build_wti_news_config,
    build_wti_news_contrarian_config,
    build_wti_news_factors_v2_config,
    build_wti_news_scenario_schema_config,
    build_wti_scenario_schema_predictor,
)
from energy_oil_forecasting.cfm_agent_v_5_2 import (
    build_cfm_agent_config,
    build_cfm_agent_predictor,
)
from energy_oil_forecasting.analyst_agent.agent_original import (
    build_wti_agent_predictor as build_wti_agent_predictor_original,
    build_wti_news_config as build_wti_news_config_original,
)
from energy_oil_forecasting.cfm_agent_v_5_2_2_delta_governed import (
    build_cfm_agent_config_delta_governed,
    build_cfm_agent_predictor_delta_governed,
)
from energy_oil_forecasting.scenario_schema_anchored import (
    build_wti_news_scenario_schema_anchored_config,
    build_wti_scenario_schema_anchored_predictor,
)
from energy_oil_forecasting.prophet_baseline import ProphetPredictor


@dataclass
class PredictorEntry:
    """One row in the experiment. Flip ``enabled`` to switch a predictor on/off."""

    name: str
    factory: Callable[[], object]  # lazy — built only when enabled
    enabled: bool = True
    baseline: bool = False  # baselines are saved to curriculum/ for NB05–06


# LLM / agent factories — each takes a model so the same recipe runs on both.
# LLMP-Sampled optionally serializes the covariate panel into the prompt
# (labeled exogenous-series blocks); the others are target-only. A distinct
# variant_tag keeps the +cov run separate in the cache and on the leaderboard.
def _llmp_sampled(model, covariates=None):
    return SampledTrajectoryLLMPredictor(
        SampledTrajectoryLLMPredictorConfig(
            model=model,
            n_samples=N_SAMPLES,
            covariate_series_ids=covariates,
            variant_tag="cov" if covariates else None,
        )
    )


def _llmp_grid(model):
    return QuantileGridLLMPredictor(QuantileGridLLMPredictorConfig(model=model))


#def _news_agent(model):
#    return build_wti_agent_predictor(build_wti_news_config(model=model))


#def _news_agent_contrarian(model):
#    return build_wti_agent_predictor(build_wti_news_contrarian_config(model=model))


def _news_agent(model):
    return build_wti_agent_predictor(
        build_wti_news_config(model=model, verifier_max_attempts=1, verifier_confidence_threshold=6
                              )
    )


def _news_agent_contrarian(model):
    return build_wti_agent_predictor(
        build_wti_news_contrarian_config(model=model, verifier_max_attempts=1, verifier_confidence_threshold=6
                                         )
    )


def _news_agent_factors_v2(model):
    return build_wti_agent_predictor(
        build_wti_news_factors_v2_config(model=model, verifier_max_attempts=1, verifier_confidence_threshold=6
                                         )
    )

def _news_agent_scenario_schema(model):
    return build_wti_scenario_schema_predictor(
        build_wti_news_scenario_schema_config(model=model)
    )

def _news_agent_original(model):
    return build_wti_agent_predictor_original(build_wti_news_config_original(model=model))

def _news_agent_scenario_schema_zero_temp(model):
    # Base Scenario Schema with ONLY temperature pinned -- no other change.
    # The config builder hardcodes name="wti_analyst_news_scenario_schema", and
    # predictor_id is agent_predictor_{name}_{model}_{modality} with no
    # temperature component, so without renaming this would resolve to the same
    # id as the base entry and the two would overwrite each other's cache.
    config = build_wti_news_scenario_schema_config(model=model, temperature=0.0)
    config = config.model_copy(update={"name": "wti_analyst_news_scenario_schema_temp0"})
    return build_wti_scenario_schema_predictor(config)


def _news_agent_scenario_schema_enhanced(model):
    from energy_oil_forecasting.scenario_schema_enhanced import (
        build_wti_news_scenario_schema_enhanced_config,
        build_wti_scenario_schema_enhanced_predictor,
    )
    config = build_wti_news_scenario_schema_enhanced_config(model=model)
    return build_wti_scenario_schema_enhanced_predictor(config)


def _cfm_agent_v_5_2():
    config = build_cfm_agent_config()
    return build_cfm_agent_predictor(config)


def _cfm_agent_v_5_2_arima_only():
    from energy_oil_forecasting.cfm_agent_v_5_2 import (
        build_cfm_agent_config_arima_only,
        build_cfm_agent_predictor_arima_only,
    )
    config = build_cfm_agent_config_arima_only()
    return build_cfm_agent_predictor_arima_only(config)


def _cfm_agent_v_5_2_2_delta_governed():
    config = build_cfm_agent_config_delta_governed()
    return build_cfm_agent_predictor_delta_governed(config)


def _news_agent_scenario_schema_anchored(model, *, log_returns: bool):
    # The flag is passed to BOTH builders on purpose. The config builder
    # uses it to pick the agent name (and so the predictor_id and cache
    # file); the predictor builder uses it to pick the actual ARIMA
    # specification. Passing it in one place only would produce a cache
    # whose name does not describe what ran.
    config = build_wti_news_scenario_schema_anchored_config(model=model, log_returns=log_returns)
    return build_wti_scenario_schema_anchored_predictor(config, anchor_log_returns=log_returns)


# ── Experiment registry ───────────────────────────────────────────────────────
# Toggle `enabled` on any line to include/exclude that predictor.
#
# The enabled/disabled flags below are evidence-driven, not habit. Every number
# quoted comes from ONE origin grid (backtest 2014-04-14 → 2024-07-23, 43
# origins / 167 predictions; eval 2024-07-23 → 2026-04-27, 8 origins), scored by
# scripts/run_autoarima_agent_grid.py. Mean CRPS across all horizons, and the
# paired difference against AutoARIMA (negative = beats it):
#
#                              backtest  eval    vs AutoARIMA, backtest / eval
#   News Agent Scenario Schema   3.65    4.21    -0.62 p<0.0001 / -0.70 p=0.0028
#   News Agent Scenario          3.78    5.17    -0.49 p<0.0001 / +0.26 p=0.86
#   News Agent (original)        3.80    4.65    -0.47 p<0.0001 / -0.26 p=0.42
#   AutoARIMA                    4.26    4.91    —
#   ECM (base)                   4.34    4.95    +0.08 p<0.0001 / +0.04 p=0.06
#   Kalman                       4.98    5.73    +0.71 p=0.0003  / +0.82 p=0.20
#   Naive                        5.75    6.62    +1.48 / +1.71
#   LightGBM + cov               loses to Naive at every horizon (own grid)
#
# Read the eval column, not the backtest one. Every agent beats AutoARIMA
# in-sample; only Scenario Schema still does out-of-sample.
from aieng.forecasting.methods.numerical.darts_classical import DartsKalmanForecasterPredictor

REGISTRY = [
    # Floor and reference points. Free, and Naive anchors every comparison.
    PredictorEntry("Naive (Last Value)", LastValuePredictor, enabled=True, baseline=True),
    PredictorEntry("Kalman", DartsKalmanForecasterPredictor, enabled=True, baseline=True),

    # Best token-free method, and the best-calibrated method in the whole
    # line-up: 78–88% coverage against a nominal 80% at every horizon.
    PredictorEntry("AutoARIMA", DartsAutoARIMAPredictor, enabled=True),

    # ON — the same AutoARIMA fitted on log RETURNS. The fitted target y is
    # diff(log(price)); sample paths are cumulated and applied to the last
    # observed close. This matches the construction in Kam's study rather than
    # the near neighbour of "fit on log prices and let AutoARIMA choose d",
    # which is equivalent only when it happens to pick d=1 and anchors on
    # ARIMA's own fitted level instead of the last close. Innovation variance
    # scales with the price level rather than being constant in dollars, which
    # matters for a series that has traded from under $20 to over $140 here.
    #
    # Kam's independent level-vs-return study finds return specifications
    # preferable within every model family, with AutoARIMA Return the strongest
    # statistical point-forecast model overall (best MAE at 1/2/4 weeks, best
    # one-week RMSE, correlation and CRPS), though its edge over the benchmarks
    # narrows materially beyond one week. That study reports calibration
    # explicitly for Kalman but not for ARIMA, and calibration is what this grid
    # is set up to measure: the level version above holds 80.5/80.5/81.0/88.4%
    # coverage at h=5/10/21/63 against a nominal 80%, which is the best in the
    # line-up and is what both anchored agents build on. Whether the return
    # version keeps that is the open question, so both run side by side here
    # rather than one replacing the other.
    #
    # The agents deliberately still anchor on the level version this run.
    # Switching their anchor at the same time as testing whether they preserve
    # ARIMA's calibration would change two things at once.
    PredictorEntry(
        "AutoARIMA (log returns)",
        lambda: DartsAutoARIMAPredictor(log_returns=True),
        enabled=True,
    ),

    # OFF — redundant with AutoARIMA. Ties it on CRPS (+0.08 backtest, +0.04
    # eval) while winning under a third of paired points, and needs a
    # seven-series covariate panel to do it. Its four "expanded" variants are
    # not reproducible here: they need nine series this data.py lacks, and the
    # expanded/levelonly pair records byte-identical metadata, so whatever
    # separated them was never committed. On the local grid expansion moved
    # CRPS by 0.06, so little is lost.
    # ON — token-free, so it can be scored without spending anything on the
    # LLM entries. It was switched off on the earlier grid as redundant with
    # AutoARIMA (+0.08 backtest / +0.04 eval CRPS, winning under a third of
    # paired points), but that judgement predates the calibration work: the
    # coverage question this grid exists to answer was never asked of it.
    #
    # predictor_id is 'ecm_regression', which nothing else on this grid
    # uses, so it writes a fresh cache file and cannot collide.
    PredictorEntry(
        "ECM (base)",
        lambda: ErrorCorrectionRegressionPredictor(covariate_series_ids=COVARIATES),
        enabled=True,
    ),

    # ON — the log-target counterpart, and the ECM analogue of what the
    # AutoARIMA pair tests. NOT use_log_levels: that logs the covariates too,
    # and six of these seven go negative (the five _log_ret_ series on any down
    # day, and oil_curve_contango is log(USL/USO), negative under backwardation)
    # -- so it refuses to run. log_target leaves X alone.
    #
    # predictor_id is 'ecm_regression_logtgt', so it cannot collide with the
    # level entry above, and both are numerical: this pair costs no tokens.
    PredictorEntry(
        "ECM (log target)",
        lambda: ErrorCorrectionRegressionPredictor(covariate_series_ids=COVARIATES, log_target=True),
        enabled=True,
    ),

    # ON — the expanded panel: the seven originals plus nine level-valued
    # series (OVX, 3-2-1 crack spread, 10y yield, Brent price, copper, S&P
    # 500, dollar-index level, EIA crude stocks, EIA refinery utilization),
    # and the level-only split, and its log-target counterpart. All numerical,
    # so this trio costs no tokens.
    #
    # These reconstruct the STRUCTURE of the old expanded/levelonly variants,
    # not their data. The originals used nine EIA and FRED series that data.py
    # does not define and whose caches are not on this machine. They are also
    # not a clean experiment even with the data: fredgraph.csv serves today's
    # revised numbers, so a 2014-2024 backtest reading INDPRO would see
    # revisions published years after the origin, and STLFSI4 is worse still --
    # a recomputed index that replaced STLFSI2 and STLFSI3, so the series does
    # not merely differ from what was known then, it did not exist. Point-in-
    # time vintages need a FRED API key.
    #
    # Everything used here is a market price, which is never revised: the value
    # quoted on a given day is final, so no vintage problem exists.
    #
    # variant_tag keeps the ids distinct (ecm_regression_expanded_yh2...) and
    # deliberately does NOT reuse the old expanded/expanded_levelonly names,
    # which denote a different fourteen-series panel.
    #
    # The tag is _yh2, not _yh, because the panel changed: Brent price level was
    # added after the first run showed the error-correction term was being zeroed
    # on 90.4% of origins with a median Engle-Granger p of 0.659. An ECM needs
    # covariates that are I(1) and share a stochastic trend with the target, and a
    # panel of log RETURNS cannot supply one -- returns are stationary. WTI and
    # Brent are the textbook cointegrating pair, so Brent's LEVEL is the anchor
    # that was missing; its return stays in the short-run equation.
    #
    # The _yh results remain on disk, so this is also the clean A/B on whether
    # a panel of genuine price levels rescues the cointegration or the model is
    # beyond help.
    #
    # Variable choices follow the literature rather than what Yahoo happens to
    # serve. Copper is the daily stand-in for Kilian's (2009) aggregate-demand
    # channel; crude inventories are the state variable in Kilian & Murphy
    # (2014) and the most-cited fundamental in the field; the dollar level is
    # Akram (2009) and Chen, Rogoff & Rossi (2010). CPI was considered and
    # rejected: at daily frequency it is nearly a deterministic trend, and
    # adding a near-trend to a cointegrating regression is a known route to
    # SPURIOUS cointegration -- which is the one failure this panel change
    # must not manufacture, given the problem being fixed is a zeroed
    # error-correction term.
    #
    # Two caveats. The EIA pair needs EIA_API_KEY and is skipped without it,
    # so the panel silently shrinks on a machine that has no key -- check
    # n_selected and the covariates list in the metadata before comparing runs
    # across machines. And _aligned_frame inner-joins every series, so the fit
    # is restricted to sessions where all of them are observed; watch
    # n_observations against the base ECM's ~4125.
    PredictorEntry(
        "ECM (expanded)",
        lambda: ErrorCorrectionRegressionPredictor(
            covariate_series_ids=COVARIATES_EXPANDED, variant_tag="expanded_yh2"
        ),
        enabled=True,
    ),

    # The control that isolates what the level-only split actually does: same
    # panel as the entry above, only the long-run/short-run allocation differs.
    # Levels are what can cointegrate with the target; the other five series are
    # already log returns, i.e. already differenced, so they belong in the
    # short-run equation and nowhere else.
    PredictorEntry(
        "ECM (expanded, level-only)",
        lambda: ErrorCorrectionRegressionPredictor(
            covariate_series_ids=COVARIATES_EXPANDED,
            long_run_only_covariate_series_ids=COVARIATES_LEVEL_ONLY,
            variant_tag="expanded_yh2_levelonly",
        ),
        enabled=True,
    ),

    PredictorEntry(
        "ECM (expanded, level-only, log target)",
        lambda: ErrorCorrectionRegressionPredictor(
            covariate_series_ids=COVARIATES_EXPANDED,
            long_run_only_covariate_series_ids=COVARIATES_LEVEL_ONLY,
            log_target=True,
            variant_tag="expanded_yh2_levelonly",
        ),
        enabled=True,
    ),

    # ON — a single-variable A/B against "ECM (level)": same panel, same target,
    # only the assumed future covariate path differs. The default "zero" freezes
    # every covariate at its last observed level for the whole horizon, which for
    # a 63-day forecast is a strong claim nobody has tested.
    #
    # Expect this to lose, and badly. "last" repeats the most recent observed
    # DIFFERENCE at every step, and five of the seven covariates are already log
    # returns -- so x is a return near 0.012 and dx is the change in that return,
    # near 0.017. Repeating it 63 times drifts the covariate to roughly 1.08, a
    # 108% daily return. The class docstring warns about this for 21 steps; at 63
    # it should be worse.
    #
    # Run anyway because it is free and it is the only way to find out whether
    # "zero" is actually the better assumption or merely the untested default. If
    # both look bad the useful conclusion is that neither constant path is right
    # and the covariate path needs its own forecast -- which is what a full VECM
    # would give and this single-equation model cannot.
    PredictorEntry(
        "ECM (level, diff-path=last)",
        lambda: ErrorCorrectionRegressionPredictor(
            covariate_series_ids=COVARIATES, covariate_diff_path="last", variant_tag="difflast"
        ),
        enabled=True,
    ),

    # OFF — loses to Naive at every horizon, nothing significant, for
    # ~185 s/origin. Naive is the floor AutoARIMA clears by 1.48 CRPS.
    PredictorEntry(
        "LightGBM + cov",
        lambda: DartsLightGBMPredictor(
            lags=LAGS, lags_past_covariates=LAGS, covariate_series_ids=COVARIATES,
            num_samples=NUM_SAMPLES_LGBM, lgbm_kwargs=LGBM_KWARGS,
        ),
        enabled=False,
    ),

    PredictorEntry(f"News Agent ({LITE_MODEL})", lambda: _news_agent(LITE_MODEL), enabled=False),

    # ON — the unmodified Vector agent, kept as the "are we beating the
    # original" yardstick. Its own numbers argue against paying for it (eval
    # -0.26 CRPS, p=0.42; the in-sample win does not survive), but that is the
    # point: every variant in this file descends from it, and without it on the
    # same grid the comparison is against a number from a different run.
    PredictorEntry(
        f"News Agent Original ({LITE_MODEL})",
        lambda: _news_agent_original(LITE_MODEL),
        enabled=True,
    ),

    # ON — the only agent that earns its tokens. Beats AutoARIMA by ~14% on
    # both windows and is the sole method holding its edge out of sample (84%
    # of paired eval points). Its weakness is calibration: 48–67% coverage
    # against a nominal 80%, so it wins on median accuracy while being
    # overconfident. Widening its intervals is the open lever.
    PredictorEntry(
        f"News Agent Scenario Schema ({LITE_MODEL})",
        lambda: build_wti_scenario_schema_predictor(build_wti_news_scenario_schema_config(model=LITE_MODEL)),
        enabled=True,
    ),

    # ON — base Scenario Schema with temperature pinned to 0 and nothing else
    # changed. Exists to disentangle the Enhanced entry below: base -> this
    # isolates temperature, this -> Enhanced isolates Enhanced's extra prompt
    # text. Without it, Enhanced differs from base in two ways at once and a
    # win cannot be attributed to either.
    PredictorEntry(
        f"News Agent Scenario Schema - zero temp ({LITE_MODEL})",
        lambda: _news_agent_scenario_schema_zero_temp(LITE_MODEL),
        enabled=True,
    ),

    # ON — the third Scenario Schema variant, so all three run on one grid.
    # Best performer in the 2026 eval (7.78 CRPS, 60% coverage, top of all
    # eight), but note what actually differs from the base entry above: its
    # headline "memory" feature is inert. The prompt tells the agent to check
    # for previous scenario frameworks in its context, while the predictor uses
    # the same WtiPriceForecastPromptBuilder as the base agent, whose payload
    # carries only task/as_of/horizons/quantiles/summary/history. Nothing ever
    # supplies prior frameworks, so the instruction always takes its own "if no
    # prior frameworks are available, proceed with standard analysis" branch.
    #
    # Two consequences. No cross-origin state means no leakage risk in a
    # backtest. And the real difference from the base variant is temperature=0
    # (base uses the model default) plus inert prompt text -- so if Enhanced
    # wins again here, temperature pinning is the more parsimonious
    # explanation than memory, and nb04's disabled zero-temp base entry is what
    # would separate the two.
    PredictorEntry(
        f"News Agent Scenario Schema Enhanced - temp=0 ({LITE_MODEL})",
        lambda: _news_agent_scenario_schema_enhanced(LITE_MODEL),
        enabled=True,
    ),

    # CFM Agent v5.2: policy-controlled forecaster with gemini-3.1-flash-lite
    # optimization (2026.08.19). Compare against Scenario Schema on quarterly grid
    # (same origin dates for fair comparison).
    # OFF — its ensemble runs Kalman + LightGBM + ARIMA, and the LightGBM fits
    # dominate runtime on a 43-origin grid. Superseded as a control by the
    # ARIMA-only variant below, which is both faster and a cleaner comparison
    # against Delta-Governed.
    PredictorEntry(
        "CFM Agent v5.2",
        _cfm_agent_v_5_2,
        enabled=False,
    ),

    # ON — the control for Delta-Governed, and the reason CFM v5.2 above can
    # stay off. Delta-Governed IS this agent with a different governor: same
    # ARIMA-only ensemble, same evidence-tier gating, same prompt scaffolding
    # and research pipeline. The only differences are the three governor
    # mechanics (discrete rank vs named category; historical-percentile shift
    # vs fixed fraction of ensemble width; historical spread vs ensemble spread
    # for the interval). So this pair is a clean A/B on exactly the change
    # under test, whereas CFM v5.2 differs in both the governor AND the
    # ensemble (Kalman+LightGBM+ARIMA) and would have been confounded even if
    # its LightGBM fits were free -- which they are not; stripping them is
    # what market_data_arima_only.py exists for.
    #
    # Without this control, a well-calibrated Delta-Governed result would be
    # unattributable: the governance working, or CFM-family agents simply being
    # well calibrated.
    PredictorEntry(
        "CFM Agent v5.2.2 (ARIMA only)",
        _cfm_agent_v_5_2_arima_only,
        enabled=True,
    ),

    # ON — the two agents built to stop the LLM inventing numbers, on the only
    # grid long enough to test that claim. The open question this run exists to
    # answer is calibration, not CRPS: over 2014-2024 AutoARIMA holds 80.5 /
    # 80.5 / 81.0 / 88.4% coverage against a nominal 80% at h=5/10/21/63, while
    # News Agent Scenario Schema -- built on that same AutoARIMA -- gets 56.1 /
    # 53.7 / 69.0 / 62.8%. Its misses are balanced (33 above, 33 below) with
    # near-zero bias, so it is not mis-pointing; it narrows intervals it has no
    # basis to narrow, which is the "widening its intervals is the open lever"
    # note above, measured.
    #
    # Delta-Governed gates the LLM to a discrete rank clipped by evidence tier,
    # and Anchored pins the centre to a deterministic AutoARIMA anchor shifted
    # only as far as real historical price moves justify. Both exist to prevent
    # exactly that narrowing. Whether they actually preserve AutoARIMA's
    # calibration is untested outside a single 25-prediction 2026 window, which
    # has already produced two conclusions that did not survive contact with a
    # second regime.
    PredictorEntry(
        "CFM Agent v5.2.2 Delta-Governed",
        _cfm_agent_v_5_2_2_delta_governed,
        enabled=True,
    ),
    # Log-return anchor. The level version is NOT a registry entry: its
    # results are already cached under the unsuffixed agent name, and the
    # analysis scripts (miss_structure.py, width_recalibration.py) read the
    # prediction directory rather than this registry, so the comparison is
    # available without paying for a re-run. Set log_returns=False here if
    # you want it back on the notebook leaderboard -- it is a cache hit.
    #
    # Worth measuring on this run: the offline swap that motivated the
    # change (scripts/compare_anchor_level_vs_logret.py) held the LLM's
    # cached scenarios fixed and swapped only the post-call arithmetic.
    # AnchoredPromptBuilder puts the anchor IN the prompt, so the LLM now
    # reads different numbers too -- an effect that comparison could not
    # see, and the reason this entry is a real run rather than a re-score.
    # OFF for this pass only. It is the ONE entry with no cache, so it is the
    # only thing on this grid that would call the model; everything else
    # reloads from disk. Switched off so ECM can be scored token-free.
    # Turn back on once you have read the ECM numbers.
    PredictorEntry(
        f"News Agent SS Anchored - log returns ({LITE_MODEL})",
        lambda: _news_agent_scenario_schema_anchored(LITE_MODEL, log_returns=True),
        enabled=False,
    ),
]

# Instantiate only the enabled predictors (lazy factories skip the rest).
PREDICTORS = {e.name: e.factory() for e in REGISTRY if e.enabled}
_BASELINE_PREDICTORS = {e.name for e in REGISTRY if e.baseline}

print(f"Active predictors ({len(PREDICTORS)}):")
for name in PREDICTORS:
    tag = "  (baseline → curriculum/)" if name in _BASELINE_PREDICTORS else ""
    print(f"  {name}{tag}")

Active predictors (11):
  Naive (Last Value)  (baseline → curriculum/)
  Kalman  (baseline → curriculum/)
  AutoARIMA
  AutoARIMA (log returns)
  News Agent Original (gemini-3.1-flash-lite-preview)
  News Agent Scenario Schema (gemini-3.1-flash-lite-preview)
  News Agent Scenario Schema - zero temp (gemini-3.1-flash-lite-preview)
  News Agent Scenario Schema Enhanced - temp=0 (gemini-3.1-flash-lite-preview)
  CFM Agent v5.2.2 (ARIMA only)
  CFM Agent v5.2.2 Delta-Governed
  News Agent Scenario Schema Anchored (gemini-3.1-flash-lite-preview)


---
## 3. Run the 2025 Historical Backtest

All 51 weekly origins in 2025 are evaluated for each predictor.
`cached_multi_backtest` caches results under `data/predictions/` so
subsequent runs are instant.

In [3]:
import time

print(f"Running rolling backtest ({backtest_spec.start} → {backtest_spec.end}, {len(PREDICTORS)} predictor(s))...")
print("LLM/agent runs are expensive — first run will take several minutes.\n")

backtest_results: dict[str, object] = {}
for i, (_name, _predictor) in enumerate(PREDICTORS.items()):
    if i > 0:
        time.sleep(15)  # pace between predictors, not just after a failure
    backtest_results[_name] = cached_multi_backtest(
        _predictor, backtest_spec, data_service,
        max_retries=4,     # was 2
        retry_delay=30.0,  # was 2.0 — long enough to clear an RPM window
        force_refresh=False,
    )
    print(f"  {_name} ✓")

print(f"\nBacktest complete ({backtest_spec.start} → {backtest_spec.end}).")

Running rolling backtest (2014-04-14 → 2024-07-23, 11 predictor(s))...
LLM/agent runs are expensive — first run will take several minutes.

  Naive (Last Value) ✓
  Kalman ✓
  AutoARIMA ✓
  AutoARIMA (log returns) ✓
  News Agent Original (gemini-3.1-flash-lite-preview) ✓
  News Agent Scenario Schema (gemini-3.1-flash-lite-preview) ✓
  News Agent Scenario Schema - zero temp (gemini-3.1-flash-lite-preview) ✓
  News Agent Scenario Schema Enhanced - temp=0 (gemini-3.1-flash-lite-preview) ✓
  CFM Agent v5.2.2 (ARIMA only) ✓
  CFM Agent v5.2.2 Delta-Governed ✓
  News Agent Scenario Schema Anchored (gemini-3.1-flash-lite-preview) ✓

Backtest complete (2014-04-14 → 2024-07-23).


---
## 4. Performance Characterisation

We score every active predictor on the 2025 backtest data:
- **CRPS** (Continuous Ranked Probability Score) — sharpness + calibration combined
- **MAE at h=21d** — point forecast accuracy at the longest horizon

The leaderboard ranks the families against each other — how much structure the
numerical methods (AutoARIMA, LightGBM ± covariates) extract over the naive
floor, whether the covariate panel earns its keep, and how the LLM/agent methods
compare across the two models. Where each method wins and where it struggles in
2025 is exactly the material the adaptive agent learns from in Notebook 5.

In [4]:
import math

from energy_oil_forecasting.analysis import per_horizon_scores, score_backtest_results


HORIZONS = backtest_spec.tasks[0].horizons

leaderboard_rows = []
unmatched_total = 0

for name, results in backtest_results.items():
    # score_backtest_results already filters mae_h21 to a single horizon — it
    # matches forecast_date back to its horizon before accumulating. The
    # per-horizon call is here because this spec has four horizons and one of
    # them, h=63, is the whole point of the quarterly run: a one-horizon
    # summary would leave three columns unreported.
    #
    # Coverage is the reason this matters more than it looks. The pooled
    # figure averages h=5 with h=63, which hides exactly the effect being
    # measured — AutoARIMA (log returns), for instance, runs 80.5 / 70.7 /
    # 85.7 / 88.4% across the four and pools to a reassuring 81.4%.
    scores = score_backtest_results(results, data_service)
    by_h = per_horizon_scores(results, data_service)
    unmatched_total += int(by_h[-1]["n"])

    row = {
        "Predictor": name,
        "Mean CRPS": scores.get("mean_crps", float("nan")),
        "Cov80 (pooled)": scores.get("coverage_80", float("nan")),
    }
    for h in HORIZONS:
        row[f"MAE h={h}d"] = by_h.get(h, {}).get("mae", float("nan"))
        row[f"Cov80 h={h}d"] = by_h.get(h, {}).get("coverage_80", float("nan"))
    leaderboard_rows.append(row)

df_leaderboard = pd.DataFrame(leaderboard_rows).set_index("Predictor")
df_leaderboard = df_leaderboard.sort_values("Mean CRPS")

print("━" * 72)
print(f"BACKTEST PERFORMANCE SUMMARY ({backtest_spec.start} → {backtest_spec.end}):")
print("━" * 72)
print(df_leaderboard.to_string(float_format=lambda v: f"{v:.2f}"))
if unmatched_total:
    print(f"\n! {unmatched_total} resolved predictions matched no declared horizon and were dropped.")

# Nominal coverage is 80%. A model in the 50s-60s is not a near miss: roughly
# one observation in three lands outside a band that should miss one in five.
# Naive shows 0.0 everywhere because its quantiles are all equal to the last
# price — a zero-width band can never contain anything. That is a degenerate
# forecast, not a calibration result.
print("\nCalibration vs the 80% nominal band (percentage points off, per horizon):")
cov_cols = [f"Cov80 h={h}d" for h in HORIZONS]
print((df_leaderboard[cov_cols] - 80).to_string(float_format=lambda v: f"{v:+.1f}"))

kalman_crps = df_leaderboard.loc["Kalman", "Mean CRPS"] if "Kalman" in df_leaderboard.index else float("nan")
naive_crps = (
    df_leaderboard.loc["Naive (Last Value)", "Mean CRPS"]
    if "Naive (Last Value)" in df_leaderboard.index
    else float("nan")
)
if not math.isnan(kalman_crps) and not math.isnan(naive_crps):
    print(
        f"\nKalman CRPS improvement over Naive: {naive_crps - kalman_crps:.4f} "
        f"({(naive_crps - kalman_crps) / naive_crps:.1%})"
    )

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BACKTEST PERFORMANCE SUMMARY (2014-04-14 → 2024-07-23):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                                                              Mean CRPS  Cov80 (pooled)  MAE h=5d  Cov80 h=5d  MAE h=10d  Cov80 h=10d  MAE h=21d  Cov80 h=21d  MAE h=63d  Cov80 h=63d
Predictor                                                                                                                                                                                            
News Agent Scenario Schema - zero temp (gemini-3.1-flash-lite-preview)             3.50           61.08      2.89       60.98       3.89        51.22       4.01        66.67       8.20        65.12
News Agent Scenario Schema Enhanced - temp=0 (gemini-3.1-flash-lite-preview)       3.54           61.96      2.92       62.50       3.90        60.00       4.07        65.85       8.53        59.52
News A

In [ ]:
from energy_oil_forecasting import viz
from energy_oil_forecasting.analysis import leaderboard_with_uncertainty, predictions_to_frame


# Same leaderboard, now with a standard-error bar on each mean. If the bars of
# the top methods overlap, their ordering is not statistically distinguishable.
#
# This is the backtest twin of the eval chart in Section 7, and it is the more
# informative of the two: 43 origins here against the eval window's 8, so the
# bars are roughly 2.3x tighter and can separate methods the eval chart cannot.
# Read them together rather than either alone -- a gap that clears the noise
# here but not there is an in-sample edge that has not been shown to survive,
# which is the pattern most of this line-up follows.
backtest_frame = predictions_to_frame(backtest_results, data_service)
backtest_board = leaderboard_with_uncertainty(backtest_frame)
viz.make_leaderboard_interval_chart(backtest_board)

In [5]:
# ── Skipped: shared curriculum/ files feed NB05 ("explores 2025 data") and
# NB06 — this notebook's backtest_spec is the 10yr/quarterly window, not 2025
# weekly, so writing here would silently corrupt their training input.
#
# _CURRICULUM_DIR = Path("adaptive_agent/curriculum")
# _CURRICULUM_DIR.mkdir(exist_ok=True)
# for _name, _result_dict in backtest_results.items():
#     if _name not in _BASELINE_PREDICTORS:
#         continue
#     _result = next(iter(_result_dict.values()))
#     (_CURRICULUM_DIR / f"backtest_{_name}.json").write_text(_result.model_dump_json(), encoding="utf-8")
# print(f"Saved {sum(n in _BASELINE_PREDICTORS for n in backtest_results)} backtest result(s) to {_CURRICULUM_DIR}/")

In [6]:
from collections import Counter

from energy_oil_forecasting.analysis import _business_horizon


# Grid check: every predictor must have been scored on the SAME origins and the
# same horizon counts. The prediction cache is keyed by spec_id alone, so a
# predictor whose results were computed on an older grid is silently reloaded
# and then compared as if it were the same experiment (see the ANCHOR_END note
# in Section 1). A mismatch here is that failure, caught.
#
# Iterates over whatever is enabled rather than a hardcoded list — naming a
# disabled predictor (e.g. "CFM Agent v5.2") used to KeyError here.
_grids = {}
for name, result_dict in backtest_results.items():
    result = next(iter(result_dict.values()))
    origins = tuple(sorted({pd.Timestamp(p.as_of).date() for p in result.predictions}))
    horizons_present = Counter(
        _business_horizon(pd.Timestamp(p.as_of), pd.Timestamp(p.forecast_date)) for p in result.predictions
    )
    _grids[name] = origins
    print(f"{name}")
    print(f"  origins: {len(origins)}  ({origins[0]} → {origins[-1]})")
    print(f"  horizon counts: {dict(sorted(horizons_present.items()))}")

_reference = next(iter(_grids.values()))
_divergent = {n: g for n, g in _grids.items() if g != _reference}
if _divergent:
    print(f"\n!! {len(_divergent)} predictor(s) were scored on a DIFFERENT origin grid:")
    for name, grid in _divergent.items():
        print(f"   {name}: {len(grid)} origins vs {len(_reference)} — their scores are not comparable.")
    print("   Delete the offending cached files under data/predictions/ and rerun.")
else:
    print(f"\nAll {len(_grids)} predictors scored on the same {len(_reference)} origins.")

Naive (Last Value)
  origins: 43  (2014-04-14 → 2024-06-04)
  horizon counts: {5: 41, 10: 41, 21: 42, 63: 43}
Kalman
  origins: 43  (2014-04-14 → 2024-06-04)
  horizon counts: {5: 41, 10: 41, 21: 42, 63: 43}
AutoARIMA
  origins: 43  (2014-04-14 → 2024-06-04)
  horizon counts: {5: 41, 10: 41, 21: 42, 63: 43}
AutoARIMA (log returns)
  origins: 43  (2014-04-14 → 2024-06-04)
  horizon counts: {5: 41, 10: 41, 21: 42, 63: 43}
News Agent Original (gemini-3.1-flash-lite-preview)
  origins: 43  (2014-04-14 → 2024-06-04)
  horizon counts: {5: 41, 10: 41, 21: 42, 63: 43}
News Agent Scenario Schema (gemini-3.1-flash-lite-preview)
  origins: 43  (2014-04-14 → 2024-06-04)
  horizon counts: {5: 41, 10: 41, 21: 42, 63: 43}
News Agent Scenario Schema - zero temp (gemini-3.1-flash-lite-preview)
  origins: 43  (2014-04-14 → 2024-06-04)
  horizon counts: {5: 41, 10: 41, 21: 42, 63: 43}
News Agent Scenario Schema Enhanced - temp=0 (gemini-3.1-flash-lite-preview)
  origins: 42  (2014-04-14 → 2024-06-04)
  h

---
## 5. 2026 Evaluation — Held-Out Test Period

We run every active predictor on **18 weekly origins spanning Feb–Jun 2026**
(`energy_oil_eval.yaml`) — the major geopolitical volatility spike not seen
during the 2025 backtest, plus its aftermath. The window runs through the
most recent origin that still fully resolves against cached WTI data (the
21-business-day horizon needs data 21 business days past the origin).

This evaluation serves two purposes:
1. **Measure out-of-sample robustness** — do the 2025 edges (statistical,
   covariate, or LLM/agent) hold under a structural regime shift?
2. **Establish the stateless baseline** that the trained adaptive agents in
   Notebook 6 are compared against. The baseline predictors' results are saved
   to `adaptive_agent/curriculum/` for Notebooks 5 and 6 to load.

In [7]:
import time

print(f"Running evaluation ({eval_spec.start} → {eval_spec.end}, {len(PREDICTORS)} predictor(s))...")
eval_results: dict[str, object] = {}
for i, (name, predictor) in enumerate(PREDICTORS.items()):
    if i > 0:
        time.sleep(15)
    eval_results[name] = cached_multi_backtest(
        predictor, eval_spec, data_service,
        max_retries=4,
        retry_delay=30.0,
        force_refresh=False,
    )
    print(f"  {name} ✓")

print(f"\nEvaluation complete ({eval_spec.start} → {eval_spec.end}).")

Running evaluation (2024-07-23 → 2026-04-27, 11 predictor(s))...
  Naive (Last Value) ✓
  Kalman ✓
  AutoARIMA ✓
  AutoARIMA (log returns) ✓
  News Agent Original (gemini-3.1-flash-lite-preview) ✓
  News Agent Scenario Schema (gemini-3.1-flash-lite-preview) ✓
  News Agent Scenario Schema - zero temp (gemini-3.1-flash-lite-preview) ✓
  News Agent Scenario Schema Enhanced - temp=0 (gemini-3.1-flash-lite-preview) ✓
  CFM Agent v5.2.2 (ARIMA only) ✓
  CFM Agent v5.2.2 Delta-Governed ✓
  News Agent Scenario Schema Anchored (gemini-3.1-flash-lite-preview) ✓

Evaluation complete (2024-07-23 → 2026-04-27).


In [8]:
# ── Skipped: this notebook's eval_spec is the 10yr/quarterly holdout, not
# the original 2026 shock-period window NB05/NB06 expect from these files.
# Writing here would silently overwrite their curriculum inputs.
#
# for _name, _result_dict in eval_results.items():
#     if _name not in _BASELINE_PREDICTORS:
#         continue
#     _result = next(iter(_result_dict.values()))
#     (_CURRICULUM_DIR / f"eval_{_name}.json").write_text(_result.model_dump_json(), encoding="utf-8")
# print(f"Saved {sum(n in _BASELINE_PREDICTORS for n in eval_results)} eval result(s) to {_CURRICULUM_DIR}/")

---
## 6. Scorecard

Out-of-sample performance of every active predictor on the 2026 eval period.
These numbers are the **stateless baseline** the adaptive agent variants must
beat in Notebook 6 to demonstrate that training added value.

In [9]:
from energy_oil_forecasting.analysis import per_horizon_scores, score_backtest_results


EVAL_HORIZONS = eval_spec.tasks[0].horizons

scorecard_rows = []
unmatched_total = 0

for name in PREDICTORS:
    if name not in eval_results:
        continue
    results = eval_results[name]
    # Same split as the backtest leaderboard: score_backtest_results for the
    # overall CRPS and pooled coverage, per_horizon_scores for the breakdown.
    # The breakdown matters more here, not less — this window is only 8
    # origins, so a pooled coverage figure is averaging four already-thin
    # samples together.
    scores = score_backtest_results(results, data_service)
    by_h = per_horizon_scores(results, data_service)
    unmatched_total += int(by_h[-1]["n"])

    row = {
        "Predictor": name,
        "Mean CRPS": scores.get("mean_crps", float("nan")),
        "Cov80 (pooled)": scores.get("coverage_80", float("nan")),
    }
    for h in EVAL_HORIZONS:
        row[f"MAE h={h}d"] = by_h.get(h, {}).get("mae", float("nan"))
        row[f"Cov80 h={h}d"] = by_h.get(h, {}).get("coverage_80", float("nan"))
        row[f"n h={h}d"] = by_h.get(h, {}).get("n", float("nan"))
    scorecard_rows.append(row)

df_scorecard = pd.DataFrame(scorecard_rows).set_index("Predictor")
df_scorecard = df_scorecard.sort_values("Mean CRPS")

print("━" * 72)
print(f"EVAL SCORECARD ({eval_spec.start} → {eval_spec.end}):")
print("━" * 72)
print(df_scorecard.to_string(float_format=lambda v: f"{v:.2f}"))
if unmatched_total:
    print(f"\n! {unmatched_total} resolved predictions matched no declared horizon and were dropped.")

# The n columns are the point of this printout. At ~8 origins a single
# horizon's coverage moves 12.5 points per observation, so treat any ranking
# here as provisional and read the backtest table for calibration.
print("\nCalibration vs the 80% nominal band (percentage points off, per horizon):")
print((df_scorecard[[f"Cov80 h={h}d" for h in EVAL_HORIZONS]] - 80).to_string(float_format=lambda v: f"{v:+.1f}"))

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EVAL SCORECARD (2024-07-23 → 2026-04-27):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                                                              Mean CRPS  Cov80 (pooled)  MAE h=5d  Cov80 h=5d  n h=5d  MAE h=10d  Cov80 h=10d  n h=10d  MAE h=21d  Cov80 h=21d  n h=21d  MAE h=63d  Cov80 h=63d  n h=63d
Predictor                                                                                                                                                                                                                               
News Agent Original (gemini-3.1-flash-lite-preview)                                3.81           65.62      2.21       75.00    8.00       2.82        75.00     8.00       4.25        75.00     8.00      11.34        37.50     8.00
News Agent Scenario Schema Anchored (gemini-3.1-flash-lite-preview)                4.12           96.43      1.95

---
## 7. Diagnostics — reading past the leaderboard

The scorecard above is a single number per method. That hides *where* the score
comes from and *whether the ranking is even real*. The next cells decompose it
straight from the eval predictions — so they recompute on any rerun, smoke or
full:

- **CRPS by horizon** — does a method win everywhere, or is its mean dominated by
  one horizon? (For a short forecast, the 5-day calls are easy and nearly tied;
  the ranking is usually decided by the longest horizon.)
- **Mean CRPS ± standard error** — with only a handful of origins, are the gaps
  between methods bigger than the noise, or is the "winner" a coin flip?

With the **smoke spec (2 origins → a few scored points)** expect wide error bars
and an unstable ranking. That is exactly why a surprising leaderboard here is not
yet evidence of anything — it is a pipeline check.

In [10]:
from energy_oil_forecasting import viz
from energy_oil_forecasting.analysis import (
    build_price_frame,
    eval_narrative_md,
    extract_agent_rationales,
    leaderboard_with_uncertainty,
    per_horizon_crps,
    predictions_to_frame,
)
from IPython.display import HTML, Markdown, display  # noqa: A004


# Explode every scored 2026 eval prediction into one tidy row per
# (predictor, origin, horizon): point, 80% interval, realised price, and CRPS.
# Everything in Sections 7–10 reads from this frame, so it all recomputes when
# you flip SMOKE_TEST off and rerun.
price_df = build_price_frame(data_service)
eval_frame = predictions_to_frame(eval_results, data_service)
eval_board = leaderboard_with_uncertainty(eval_frame)
ph_crps = per_horizon_crps(eval_frame)

print("━" * 72)
print("MEAN CRPS BY PREDICTOR × HORIZON (lower = better; 'All' = overall mean):")
print("━" * 72)
print(ph_crps.round(2).to_string())

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MEAN CRPS BY PREDICTOR × HORIZON (lower = better; 'All' = overall mean):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                                                              h=5d  h=10d  h=21d  h=63d   All
predictor                                                                                                    
News Agent Original (gemini-3.1-flash-lite-preview)                           1.60   2.00   2.66   8.97  3.81
News Agent Scenario Schema Anchored (gemini-3.1-flash-lite-preview)           1.57   2.17   3.07   9.66  4.12
News Agent Scenario Schema - zero temp (gemini-3.1-flash-lite-preview)        2.08   2.91   2.74  10.60  4.59
News Agent Scenario Schema Enhanced - temp=0 (gemini-3.1-flash-lite-preview)  2.18   3.03   2.69  11.16  4.76
News Agent Scenario Schema (gemini-3.1-flash-lite-preview)                    2.08   3.08   2.88  11.12  4.79
AutoARIMA (

In [11]:
# Heatmap of the table above. Read it left-to-right: the short-horizon columns
# are usually a near-uniform green (everyone is right), and one long-horizon
# column carries the colour spread that sets the 'All' ranking.
viz.make_crps_heatmap(ph_crps)

In [12]:
# Same leaderboard, now with a standard-error bar on each mean. If the bars of
# the top methods overlap, their ordering is not statistically distinguishable —
# the honest verdict when only a few origins have been scored.
viz.make_leaderboard_interval_chart(eval_board)

---
## 8. What are the top methods actually forecasting?

A CRPS number doesn't show *behaviour*. Below, each leading method's **median
forecast and 80% interval** are drawn against the realised WTI path at every
eval origin. This is where the leaderboard becomes legible — watch for who
tracks the move, who simply anchors to the last price, and whose intervals are
too narrow to cover the outcome when the market jumps.

In [13]:
# Plot the leaderboard's top methods, and always include the best LLM/agent
# method for contrast (so the chart compares families even when a baseline leads).
_leaders = list(eval_board.index[:3])
_best_llm = next((p for p in eval_board.index if eval_board.loc[p, "family"] == "LLM / Agent"), None)
if _best_llm and _best_llm not in _leaders:
    _leaders.append(_best_llm)
print(f"Showing: {', '.join(_leaders)}")
viz.make_eval_forecast_chart(eval_frame, price_df, _leaders)

Showing: News Agent Original (gemini-3.1-flash-lite-preview), News Agent Scenario Schema Anchored (gemini-3.1-flash-lite-preview), News Agent Scenario Schema - zero temp (gemini-3.1-flash-lite-preview)


---
## 9. Reading the agent's reasoning

The news-reading agent attaches a free-text **rationale** to every forecast, and
a link to the full **Langfuse trace**. These are pulled straight from the
prediction metadata. This is where a surprising score becomes interpretable: you
can read whether the agent actually saw the geopolitical risk, and *how* it
turned that into a price and an interval — including, often, an interval far too
narrow for a regime shift.

In [14]:
# One card per (agent, origin): the rationale, the per-horizon note, and a link
# to the full reasoning trace. Empty only if no LLM/agent predictor is enabled.
eval_rationales = extract_agent_rationales(eval_results)
display(HTML(viz.render_rationales_html(eval_rationales)))

---
## 10. Takeaways — computed from this run

The summary below is **generated from the eval results in memory, not
hard-coded**, so it always matches what actually ran: the real winner, whether
its lead clears the noise floor, the horizon that decided the ranking, the
best-performing family, and a calibration line. Flip `SMOKE_TEST` off, rerun,
and these takeaways update themselves with the full leaderboard.

In [15]:
display(Markdown(eval_narrative_md(eval_frame, smoke=SMOKE_TEST)))

1. **News Agent Original (gemini-3.1-flash-lite-preview)** has the best mean CRPS (3.81) on the 2026 evaluation, ahead of **News Agent Scenario Schema Anchored (gemini-3.1-flash-lite-preview)** (4.12) by 0.31 — **well within the combined standard error**, so the ranking here is not statistically distinguishable from noise.
2. The leaderboard is **decided at h=63d**, where CRPS ranges 1.1–42.7 across methods; at the short h=5d horizon the methods are nearly tied (range 0.5–7.0). A handful of long-horizon points drives the whole ranking.
3. **By family** (mean CRPS): LLM / Agent 4.56, Numerical ML 4.95, Other 5.71, Baseline 6.62. Best family this window: **LLM / Agent**.
4. **Calibration:** News Agent Original (gemini-3.1-flash-lite-preview)'s 80% interval covered 66% of outcomes (target 80%) over its 32 scored point(s). With this few, coverage this far from target is itself a small-sample artefact, not necessarily mis-calibration.
5. Based on 8 origins / 348 scored points.

---
## 11. What stateless methods can't do

Sections 7–10 score and dissect this run on its own terms. But every method here
shares one structural limit, independent of who topped the leaderboard: it is
calibrated (or prompted) **once and never updated between rounds**. That is
intentional — it creates a clean baseline — but it leaves a systematic gap:

- **No error feedback.** If a method's intervals are consistently too narrow in
  an elevated-vol regime (read the coverage line in Section 10, and the squashed
  error bars in Section 8), it keeps making the same mistake. Nothing updates its
  calibration between origins.

- **No strategy evolution.** Each prediction starts from the same prior — the
  same fitted model, or the same prompt. Resolved outcomes disappear without
  influencing future forecasts.

- **Context without memory.** Even the news agent re-reads the world each origin;
  it does not accumulate what worked. The rationales in Section 9 are written
  fresh every time, with no record of how the last one resolved.

→ **Notebook 5** introduces adaptive agents that study the 2025 backtest, record
systematic observations, and calibrate their strategies accordingly. At inference
time, each agent receives the live stateless estimate and decides how to adjust
it — applying what it learned from training.

→ **Notebook 6** evaluates whether any training approach actually improved
out-of-sample performance on the held-out 2026 data — measured against the
stateless baseline this notebook just established.